# FedDBE Server

> The core abstraction for `FedDbe` server: [https://openreview.net/forum?id=nO5i1XdUS0](https://openreview.net/forum?id=nO5i1XdUS0)

In [ ]:
#| default_exp servers.feddbe

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [2]:
#| export
from fastcore.utils import *
from fastcore.all import *
import os
import gc
import copy
from tqdm import tqdm
from loguru import logger

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from fedai.client_selector import BaseClientSelector
from fedai.data import prepare_dl
from fedai.models import create_model
from fedai.optimizers import get_optimizer

from fedai.servers.base_server import BaseServer
from fedai.utils.registery import AlgorithmRegistry
from fedai.core import get_clean_kwargs


In [ ]:
#| export
@AlgorithmRegistry.register_server("feddbe")
class ServerFedDBE(BaseServer):
    def __init__(self,
                 cfg,
                 client_selector,
                 client_cls,
                 criterion,
                 fds,
                 writer= None,
                 device= None,
                 **kwargs
                 ):  
                 
        super().__init__(cfg, client_selector, client_cls, criterion, fds, writer, device, **kwargs) 
        

In [ ]:
#| export
@patch
def client_fn(self: ServerFedDBE, id, comm_round, client_state):

    if (comm_round == 1 and client_state == {}) or client_state == {}:
        client_state['model'] = self.model.state_dict()
        client_state['global_mean'] = None
        client_state['running_mean'] = torch.zeros(self.cfg.model.hidden_dim)
        client_state['num_batches_tracked'] = torch.tensor(0, dtype=torch.long)
        client_state['client_mean'] = nn.Parameter(torch.zeros(self.cfg.model.hidden_dim))

    model = create_model(self.cfg)
    model.load_state_dict(client_state['model'])
    client_state['model'] = model

    kwargs = get_clean_kwargs(self.cfg.optimizer)
    kwargs.pop("cls", None)
    optimizer = get_optimizer(self.cfg)(model.parameters(), **kwargs)
    optimizer.load_state_dict(client_state['optimizer']) if 'optimizer' in client_state else None
    for state in optimizer.state.values():
        for k, v in state.items():
            if isinstance(v, torch.Tensor):
                state[k] = v.to(self.device)

    client_state['optimizer'] = optimizer

    kwargs.pop("lr", None)
    opt_client_mean = get_optimizer(self.cfg)([client_state['client_mean']], **kwargs)
    opt_client_mean.load_state_dict(client_state['opt_client_mean']) if 'opt_client_mean' in client_state else None
    for state in opt_client_mean.state.values():
        for k, v in state.items():
            if isinstance(v, torch.Tensor):
                state[k] = v.to(self.device)


    train_loader = prepare_dl(self.cfg.data, id, self.fds, train=True, distributed=False)
    test_loader = prepare_dl(self.cfg.data, id, self.fds, train=False, distributed=False)
    client = self.client_cls(id= id, 
                             cfg= self.cfg,
                             train_loader= train_loader,
                             test_loader= test_loader,
                             state= client_state,
                             criterion= self.criterion,
                             device= self.device,
                             t= comm_round)
    client.logger = self.logger
    return client


In [ ]:
#| export
@patch
def init_training(self: ServerFedDBE):
    len_clients_ds = {}
    global_mean = 0
    for client_id in range(self.cfg.num_clients):
        client_state = self.state_mgr.get_state(id=client_id)
        client = self.client_fn(id= client_id, comm_round= 1, client_state= client_state)
        client.fit()
        self.state_mgr.set_state(id=client_id, state=client.save_state())
        len_clients_ds[client_id] = len(client.train_loader.dataset)

        del client 
        gc.collect()
        torch.cuda.empty_cache()

    m_t = sum(len_clients_ds.values())
    for client_id in range(self.cfg.num_clients):
        n_k = len_clients_ds[client_id]
        weight = n_k / m_t
        client_running_mean = self.state_mgr.get_state(id=client_id).get('running_mean', None)
        global_mean += client_running_mean * weight

    for client_id in range(self.cfg.num_clients):
        client_state = self.state_mgr.get_state(id=client_id)
        client_state['global_mean'] = global_mean.clone()
        self.state_mgr.set_state(id=client_id, state=client_state)
        

In [ ]:
#| export
@patch
def train(self: ServerFedDBE):

    self.init_training()

    res =  []
    selected_clients = self.client_selector.select()
    for t in range(1, self.cfg.n_rounds + 1):
        lst_active_ids = selected_clients[t-1]
        len_clients_ds = {}

        for id in lst_active_ids:
            client_state = self.state_mgr.get_state(id)
            client = self.client_fn(id= id, comm_round= t, client_state= client_state)
            client.fit()
            self.logger.info(f"Client {id} finished training.")
            self.logger.info("*"*20)
            self.state_mgr.set_state(id, client.save_state())
            
            len_clients_ds[id] = len(client.train_loader.dataset)

            del client 
            gc.collect()
            torch.cuda.empty_cache()

        self.aggregate(lst_active_ids, t, len_clients_ds)#lst_active_ids, comm_round, len_clients_ds
        train_res, test_res = self.evaluate(t)
        
        train_df, test_df = self.writer.write(lst_active_ids, train_res, test_res, t) 
        res.append((train_df, test_df))
        
    self.writer.save(res)
    self.writer.finish()

    return res

### Aggregation


In [ ]:
#| export
@patch
def aggregate(self: ServerFedDBE, lst_active_ids, comm_round, len_clients_ds):
    m_t = sum(len_clients_ds.values())
    
    with torch.no_grad():
        global_model = None
        
        for id in lst_active_ids:
            client_state_dict = self.state_mgr.get_state(id).get('model', None)

            if global_model is None:
                global_model = {k: torch.zeros_like(v) for k, v in client_state_dict.items()}

            n_k = len_clients_ds[id]
            weight = n_k / m_t
            for key in global_model.keys():
                global_model[key].add_(client_state_dict[key], alpha=weight)

        self.model.load_state_dict(global_model)
        
        for id in lst_active_ids:
            client_model = self.state_mgr.get_state(id).get('model', None)
            for key in global_model.keys():
                client_model[key] = self.model.state_dict()[key]

            self.state_mgr.set_state(id, {
                    **self.state_mgr.get_state(id),
                    'model': client_model,
                })

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()